# 第 2 章 §2.3 · Routing

先用**一次模型调用**把请求分类成 3 个标签之一，再用 `RunnableBranch` 按标签选一条分支。

和第 1 章相比变的是**控制流**：从写死的顺序，变成运行时按内容决定。

⚠️ 文件名有坑：这份 notebook 叫 `(LangGraph)`，但它**没用 LangGraph**——用的是 LangChain 的 `RunnableBranch`。书里真正的 LangGraph 条件边（`StateGraph` + `add_conditional_edges`）没给完整代码。

kernel 选 `Python (agentic-book)`。数据流全景、八个关键点、两个结构性缺陷、实测记录都在 `../../my_notes/ch02_routing.md`。

In [1]:
# 【适配】common 已由 venv 的启动引导加进 sys.path（见 tools/_book_bootstrap.py）

# 【适配】原版：from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableBranch

from common import make_llm


def booking_handler(request: str) -> str:
    """模拟 Booking Agent：处理机票/酒店预订。"""
    print("\n--- DELEGATING TO BOOKING HANDLER ---")
    return f"Booking Handler processed request: '{request}'. Result: Simulated booking action."


def info_handler(request: str) -> str:
    """模拟 Info Agent：处理一般信息问答。"""
    print("\n--- DELEGATING TO INFO HANDLER ---")
    return f"Info Handler processed request: '{request}'. Result: Simulated information retrieval."


def unclear_handler(request: str) -> str:
    """兜底：请求无法归类。"""
    print("\n--- HANDLING UNCLEAR REQUEST ---")
    return f"Coordinator could not delegate request: '{request}'. Please clarify."


# 【适配】第二行被收窄了：书里原版是 "For all other general information questions"，
#   这里改成只限与订机票/酒店相关问题。后果是「与预订无关的普通知识问题」都掉进 unclear。
#   ⚠️ 行为变更没有 diff 可看：代码一个字没动，只改了一句自然语言，分类边界就平移了。
#
# 路由的三个必要条件：候选集封闭（只 3 个值）、互斥（只落一类）、单 token 可表达。
# from_messages 把 system 和 user 分成两条消息 —— 混在一起更容易被用户输入带偏。
coordinator_router_prompt = ChatPromptTemplate.from_messages([
    ("system", """Analyze the user's request and determine which specialist handler should process it.
     - If the request is related to booking flights or hotels, output 'booker'.
     - For all other general information questions about booking flights or hotels, for example, checking destination of trip planning, output 'info'.
     - If the request is unclear or doesn't fit either category, output 'unclear'.
     ONLY output one word: 'booker', 'info', or 'unclear'."""),
    ("user", "{request}"),
])

# ★ 本章新增：整条链路唯一的模型调用。`ONLY output one word` 是**请求**，不是**约束**——
#   模型可能返回 'Booker'、'booker.'，或一句解释，见下面 .strip() 的问题。
coordinator_router_chain = coordinator_router_prompt | make_llm(temperature=0) | StrOutputParser()


# ★ 本章新增 RunnablePassthrough.assign(output=...)：保留上游所有键，再追加一个 output 键。
#
# ⚠️ x["request"]["request"] 的双重嵌套不是笔误：
#   外层 x["request"] = coordinator 里 RunnablePassthrough() 透传下来的【原始输入 dict】
#   内层 ["request"]  = 真正的用户文本
#   这是 LCEL 并行打包造成的形状，读代码最容易卡在这里。
branches = {
    "booker": RunnablePassthrough.assign(
        output=lambda x: booking_handler(x["request"]["request"])
    ),
    "info": RunnablePassthrough.assign(
        output=lambda x: info_handler(x["request"]["request"])
    ),
    "unclear": RunnablePassthrough.assign(
        output=lambda x: unclear_handler(x["request"]["request"])
    ),
}

# ★ 本章新增 RunnableBranch：(谓词, 分支) 成对，最后一对只写分支 = 默认分支。
# ⚠️ 书里用 .strip()，只能去首尾空白。'booker.' / 'Booker' / "'booker'" 全部落到默认分支，
#   而默认分支是 unclear_handler —— 于是系统礼貌地回答「请你说清楚一点」。
#   不报错、日志正常、用户被无意义地反复追问。更鲁棒的归一化见 02_3_debug_lab.ipynb 第 5 节。
delegation_branch = RunnableBranch(
    (lambda x: x["decision"].strip() == "booker", branches["booker"]),
    (lambda x: x["decision"].strip() == "info", branches["info"]),
    branches["unclear"],          # 默认分支：'unclear' 以及**任何意外输出**
)

# 这个 dict 和 Ch1 的 {"specifications": ...} 是同一个机制（RunnableParallel）：
# decision 调 1 次模型做分类，request 用 Passthrough 留一份原件给下游 handler 用。
coordinator_agent = (
    {
        "decision": coordinator_router_chain,
        "request": RunnablePassthrough(),
    }
    | delegation_branch
    | (lambda x: x["output"])     # 只把结果抛出去，丢掉中间过程
)


# 这些用例没有任何断言，只 print。所以「跑完没报错」≠「路由对了」——
# 用例名里的 booker/info/unclear 是**人写的期望标签**，不是模型的决定。
# 实测：A/C 稳定符合标签；B 靠补了 "I`m planning a trip." 才进 info；
#       D 连跑 6 次 booker 3 次 / info 3 次（temperature=0 也挡不住，见 README）。
for name, request in [
    ("A booking", "Book me a flight to London."),
    ("B info", "What is the capital of Italy? I`m planning a trip."),
    ("C unclear (book label)", "Tell me about quantum physics."),
    ("D info", "Tell me the earliest flight I can catch from New York to Roma next week."),
]:
    print(f"\n=== Running: {name} ===")
    print(f"Final Result: {coordinator_agent.invoke({'request': request})}")


=== Running: A booking ===



--- DELEGATING TO BOOKING HANDLER ---
Final Result: Booking Handler processed request: 'Book me a flight to London.'. Result: Simulated booking action.

=== Running: B info ===



--- DELEGATING TO INFO HANDLER ---
Final Result: Info Handler processed request: 'What is the capital of Italy? I`m planning a trip.'. Result: Simulated information retrieval.

=== Running: C unclear (book label) ===



--- HANDLING UNCLEAR REQUEST ---
Final Result: Coordinator could not delegate request: 'Tell me about quantum physics.'. Please clarify.

=== Running: D info ===



--- DELEGATING TO INFO HANDLER ---
Final Result: Info Handler processed request: 'Tell me the earliest flight I can catch from New York to Roma next week.'. Result: Simulated information retrieval.
